In [0]:
%pip install kaggle
dbutils.library.restartPython()

In [0]:
import os

# Configura tus credenciales de Kaggle
# Obtén tu API key desde: https://www.kaggle.com/settings/account
os.environ['KAGGLE_USERNAME'] = '<TU_USERNAME>'  # Reemplaza con tu username
os.environ['KAGGLE_KEY'] = '<TU_API_KEY>'  # Reemplaza con tu API key

print('Credenciales configuradas')

In [0]:
import kaggle
from kaggle.api.kaggle_api_extended import KaggleApi

# Inicializar API
api = KaggleApi()
api.authenticate()

# Descargar el dataset
dataset_name = 'isabelocastillo/eficiencia-energetica-edificios-de-aragon'
download_path = '/tmp/kaggle_data'

os.makedirs(download_path, exist_ok=True)
api.dataset_download_files(dataset_name, path=download_path, unzip=True)

print(f'Dataset descargado en: {download_path}')

# Listar archivos descargados
import glob
files = glob.glob(f'{download_path}/*')
for file in files:
    print(f'Archivo: {file}')

In [0]:
import pandas as pd

# Leer el archivo (ajusta el nombre según lo que se haya descargado)
files = glob.glob('/tmp/kaggle_data/*.csv')
if files:
    df = pd.read_csv(files[0])
    print(f'Archivo leído: {files[0]}')
    print(f'\nDimensiones: {df.shape}')
    print(f'\nPrimeras filas:')
    display(df.head())
    print(f'\nColumnas: {list(df.columns)}')
    print(f'\nTipos de datos:')
    print(df.dtypes)
else:
    print('No se encontraron archivos CSV')

In [0]:
from pyspark.sql import SparkSession

# Convertir pandas a Spark DataFrame
spark_df = spark.createDataFrame(df)

# Crear el esquema bronze si no existe
spark.sql('CREATE SCHEMA IF NOT EXISTS workspace.bronze')

# Guardar como tabla Delta
table_name = 'workspace.bronze.eficiencia_energetica_aragon'

# Primera vez: usar overwrite para establecer el esquema
# Después: cambiar a 'append' para solo agregar datos nuevos
spark_df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(table_name)

print(f'Tabla creada exitosamente: {table_name}')
print(f'Registros guardados: {spark_df.count()}')
print('\nNota: Para agregar solo datos nuevos en futuras ejecuciones, cambia mode("overwrite") a mode("append")')